## Installing libs

In [ ]:
# !pip install tensorboard

In [ ]:
import torch
import numpy as np

train_on_gpu = torch.backends.mps.is_available()

if not train_on_gpu:
    print('MPS is not available. Training on CPU ...')
else:
    print('MPS is available! Training on GPU ...')

In [ ]:
import pickle
import numpy as np
from skimage import io

from tqdm import tqdm, tqdm_notebook
from PIL import Image
from pathlib import Path

from torchvision import transforms
from torchvision.transforms import v2

import torchsummary

from multiprocessing.pool import ThreadPool
from sklearn.preprocessing import LabelEncoder
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn

from matplotlib import colors, pyplot as plt
%matplotlib inline

import warnings

warnings.filterwarnings(action='ignore', category=DeprecationWarning)


In [ ]:
DATA_MODES = ['train', 'val', 'test']

if torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")

TRAIN_DIR = Path('../../data/simpsons/train')
TEST_DIR = Path('../../data/simpsons/testset')

NORMALIZE_MEAN = [0.485, 0.456, 0.406]
NORMALIZE_STD = [0.229, 0.224, 0.225]

RESCALE_SIZE = [128, 128]

## Downloading files

In [ ]:
# !pip install gdown

In [ ]:
# !gdown --id 1RxBQiZgRAfio2tWhEE7lzZ6IaJzLheH1 -O ../../data/simpsons/simpsons_classification.zip

In [ ]:
# !unzip -q ../../data/simpsons/simpsons_classification.zip -d ../../data/simpsons/

Look at the file structure in the train and testset folders.

The train contains data that we will use to train the model. The images of the characters are arranged in folders named after the characters. Folder names will be used as class text labels in the future.

The test set contains images for which you will need to make a prediction of the most probable class.

To access the files, we will form a list of file names for the training+validation and test samples. These are full names that include a path to the files.

In [ ]:
train_val_files = sorted(list(TRAIN_DIR.rglob('*.jpg')))
test_files = sorted(list(TEST_DIR.rglob('*.jpg')))

We will encode characters' names into the numeric labels of the class and back using `LabelEncoder`.

For the train sample, we will form a list of text labels for all images - the name of the parent directory, which is simultaneously the character's name. Let's set the numerical marks of our encoder classes using the `fit` method.

Further, we will use the `transform` method to convert text labels to numeric labels and the `inverse_transform` method to convert numeric labels to text labels.



In [ ]:
label_encoder = LabelEncoder()

train_val_labels = [path.parent.name for path in train_val_files]

label_encoder.fit(train_val_labels)

Train test split with stratification.<br>
**WHY?**

In [ ]:
from sklearn.model_selection import train_test_split

train_files, val_files = train_test_split(train_val_files, test_size=0.25, \
                                          stratify=train_val_labels)

#### Creating Datasets and Dataloaders

Useful:
https://jhui.github.io/2018/02/09/PyTorch-Data-loading-preprocess_torchvision/


We have a set of image files that we need to convert into tensors and assign numerical class labels to them. We will implement this using our custom `SimpsonsDataset` class, which will create the necessary Dataset from a list of files. 

We will proceed according to the docs (https://docs.pytorch.org/tutorials/beginner/basics/data_tutorial.html#creating-a-custom-dataset-for-your-files):
- our custom class will inherit from Dataset
- we will redefine the `__init__`, `__len__` and `__getitem__` methods, and for convenience, we will add several methods for uploading and converting images.

It is important to understand what the self.transform_images_to_tensors () method does.

`Compose` combines the following transformation sequence:
- `PILToTensor` converts `PIL Image` to a tensor with parameters in the range of $[0, 255]$ (as with all pixels in the original image)
- ToDtype converts the tensor to FloatTensor of size ($C \times H \times W$) with pixel values in the range $[0,1]$
- then `Normalize` is scaled:
$\text{input} = \frac{\text{input} - \text{mean}}{\text{std}} $, <br> where the mean and std constants are the mean and variances across channels in the ImageNet dataset
- finally, `Resize` converts images to a size of $224 \times 224$ (the dataset description indicates that the images are of different sizes as they were taken directly from the video, so they should be brought to a single size).

In [ ]:
class SimpsonsDataset(Dataset):
    def __init__(self, files, label_encoder, mode):
        super().__init__()
        self.files = sorted(files)
        self.mode = mode

        if self.mode not in DATA_MODES:
            print(f"{self.mode} is not correct; correct modes: {DATA_MODES}")
            raise NameError

        self.label_encoder = label_encoder
        self.len_ = len(self.files)

        # ✅ transform создаётся один раз (ускорение)
        if self.mode == 'train':
            self.transform = v2.Compose([
                v2.PILToTensor(),
                v2.ToDtype(torch.float32, scale=True),
                v2.Normalize(NORMALIZE_MEAN, NORMALIZE_STD),
                # v2.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1),
                v2.Resize(RESCALE_SIZE),
            ])
        else:
            self.transform = v2.Compose([
                v2.PILToTensor(),
                v2.ToDtype(torch.float32, scale=True),
                v2.Normalize(NORMALIZE_MEAN, NORMALIZE_STD),
                v2.Resize(RESCALE_SIZE),
            ])

    def __len__(self):
        return self.len_

    def __getitem__(self, index):
        x = self.load_image(self.files[index])
        x = self.transform_images_to_tensors(x)

        if self.mode == 'test':
            return x
        else:
            path = self.files[index]
            y = self.label_encoder.transform([path.parent.name]).item()
            return x, y

    def load_image(self, file):
        image = Image.open(file)
        image.load()
        return image

    def transform_images_to_tensors(self, image):
        # ✅ просто применяем готовый transform
        return self.transform(image)

In [ ]:
train_dataset = SimpsonsDataset(train_files, label_encoder=label_encoder, mode='train')
val_dataset = SimpsonsDataset(val_files, label_encoder, mode='val')

In [ ]:
batch_size = 64

In [ ]:
train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=0,   # ← ВАЖНО
    pin_memory=False
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=0,   # ← ВАЖНО
    pin_memory=False
)

loaders = {'train': train_loader, 'val': val_loader}

#### Visualize the images

In [ ]:
def imshow(inp, title=None, plt_ax=plt, default=False):
    inp = inp.detach().cpu().numpy().transpose((1, 2, 0))
    mean = np.array([0.485, 0.456, 0.406])
    std = np.array([0.229, 0.224, 0.225])
    inp = std * inp + mean
    inp = np.clip(inp, 0, 1)
    plt_ax.imshow(inp)
    if title is not None:
        plt_ax.set_title(title)
    plt_ax.grid(False)

We will check the function with an image from a loader.

In [ ]:
image_tensor, label = next(iter(train_loader))
print(f"Numerical label: {label[0]}")
print(f"Original label: {label_encoder.inverse_transform([label[0], ])}")
imshow(image_tensor[0])
plt.show()


In [ ]:
# uncomment if you have problem with pillow
# def register_extension(id, extension): Image.EXTENSION[extension.lower()] = id.upper()
# Image.register_extension = register_extension
# def register_extensions(id, extensions):
#     for extension in extensions: register_extension(id, extension)
# Image.register_extensions = register_extensions

In [ ]:
def show_images1(n_rows, n_cols, dataset):
    fig, ax = plt.subplots(nrows=n_rows, ncols=n_cols, figsize=(n_cols * 4, n_rows * 4), \
                           sharey=True, sharex=True)

    for fig_x in ax.flatten():
        random_characters = int(np.random.uniform(0, len(dataset)))
        im_val, label = dataset[random_characters]
        img_label = " ".join(map(lambda x: x.capitalize(), \
                                 label_encoder.inverse_transform([label])[0].split('_')))
        imshow(im_val.data.cpu(), title=img_label, plt_ax=fig_x)
        fig_x.set_axis_off()
    # return None

In [ ]:
def show_images(n_rows, n_cols, dataset):
    fig, ax = plt.subplots(
        nrows=n_rows,
        ncols=n_cols,
        figsize=(n_cols * 4, n_rows * 4),
        sharey=True,
        sharex=True
    )

    ax = np.array(ax).reshape(-1)

    for fig_x in ax:
        random_characters = np.random.randint(0, len(dataset))
        im_val, label = dataset[random_characters]

        img_label = " ".join(
            map(lambda x: x.capitalize(),
                label_encoder.inverse_transform([label])[0].split('_'))
        )

        imshow(im_val.data.cpu(), title=img_label, plt_ax=fig_x)
        fig_x.set_axis_off()

    plt.tight_layout()
    plt.show()

In [ ]:
show_images(n_rows=3, n_cols=4, dataset=val_dataset)

## Step 3. Building NN

Convolutions calculator:

https://madebyollin.github.io/convnet-calculator/

#### Model

In [ ]:
class SimpleCnn(nn.Module):

    def __init__(self, n_classes):
        super().__init__()
        self.conv1 = nn.Sequential(
            nn.Conv2d(in_channels=3, out_channels=8, kernel_size=3),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2)
        )
        self.conv2 = nn.Sequential(
            nn.Conv2d(in_channels=8, out_channels=16, kernel_size=3),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2)
        )
        self.conv3 = nn.Sequential(
            nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2)
        )
        self.conv4 = nn.Sequential(
            nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2)
        )
        self.conv5 = nn.Sequential(
            nn.Conv2d(in_channels=64, out_channels=96, kernel_size=3),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2)
        )

        self.out = nn.Linear(384, n_classes)

    def forward(self, x):
        x = self.conv1(x)
        x = self.conv2(x)
        x = self.conv3(x)
        x = self.conv4(x)
        x = self.conv5(x)

        x = x.view(x.size(0), -1)
        logits = self.out(x)
        return logits

*Description*:

1. Input size: $3\times 224 \times 224$
2. Size after the first layer (Conv2d + ReLU + MaxPool2d):  $8 \times 111 \times 111$
3. After the second layer: $16 \times 54 \times 54$
4. After the third laye: $32 \times 26 \times 26$
5. After the forth laye: $64 \times 12 \times 12$
6. After the fifth laye: $96 \times 5 \times 5$
7. After the FC layer: number of classes

$$\large H_{out} = \frac{H_{in} + 2P_h - K_h}{S_h} + 1$$

- $H_{in}$: input height/width
- $P_h$: padding
- $S_h$: stride
- $K_h$: kernel size

#### Training

In [ ]:
from sklearn.metrics import f1_score


def train_one_epoch(model, optim, criterion, dataloader, device):
    model.train()

    train_loss = 0
    all_preds = []
    all_labels = []

    for data, label in dataloader:
        data, label = data.to(device), label.to(device)

        optim.zero_grad()
        output = model(data)
        loss = criterion(output, label)
        loss.backward()
        optim.step()

        train_loss += loss.item()

        preds = output.argmax(dim=1)

        # Move to CPU for sklearn
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(label.cpu().numpy())

    # Compute F1 score
    f1 = f1_score(all_labels, all_preds, average='micro')
    # use 'binary', 'macro', 'weighted' depending on task

    return train_loss / len(dataloader), f1

In [ ]:
import os


@torch.no_grad()
def eval_one_epoch(model, criterion, dataloader, device, f1_best, epoch):
    model.eval()
    valid_loss = 0
    all_preds = []
    all_labels = []

    for data, label in dataloader:
        data, label = data.to(device), label.to(device)

        output = model(data)
        loss = criterion(output, label)

        valid_loss += loss.item()

        preds = output.argmax(dim=1)

        # Move to CPU for sklearn
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(label.cpu().numpy())

    # Compute F1 score
    f1 = f1_score(all_labels, all_preds, average='micro')
    # use 'binary', 'macro', 'weighted' depending on task
    if f1 > f1_best:
        # print('Saving..')
        state = {
            'net': model.state_dict(),
            'acc': f1,
            'epoch': epoch,
        }
        if not os.path.isdir('checkpoint'):
            os.mkdir('checkpoint')
        torch.save(state, './checkpoint/ckpt_simsons.pth')

    return valid_loss / len(dataloader), f1

In [ ]:
model_simple_cnn = SimpleCnn(n_classes=len(np.unique(train_val_labels)))

model_simple_cnn.to("cpu")
torchsummary.summary(model_simple_cnn, (3, 128, 128))

model_simple_cnn.to(DEVICE)  # вернуть обратно на MPS

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model_simple_cnn.parameters(), lr=3e-4)

In [ ]:
train_losses = []
test_losses = []
best_acc = 0
best_epoch = -1
EPOCHS = 10

for i in range(1, EPOCHS + 1):
    train_loss, train_acc = train_one_epoch(model_simple_cnn, optimizer, criterion, train_loader, DEVICE)
    train_losses.append(train_loss)

    test_loss, test_acc = eval_one_epoch(model_simple_cnn, criterion, val_loader, DEVICE, best_acc, i)
    test_losses.append(test_loss)

    if test_acc > best_acc:
        best_acc = test_acc
        best_epoch = i

plt.figure(figsize=(18, 9))
plt.plot(np.arange(len(train_losses)), train_losses, label='Train Loss')
plt.plot(np.arange(len(test_losses)), test_losses, label='Test Loss')
plt.title('Loss')
plt.legend(loc='best')
plt.show()

In [ ]:
test_acc

#### Adding TensorBoard

In [ ]:
from torch.utils.tensorboard import SummaryWriter
import os

In [ ]:
model_simple_cnn = SimpleCnn(n_classes=len(np.unique(train_val_labels)))

# временно на CPU
model_simple_cnn.to("cpu")

torchsummary.summary(model_simple_cnn, (3, 128, 128))

# вернуть обратно
model_simple_cnn.to(DEVICE)

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model_simple_cnn.parameters(), lr=3e-4)

In [ ]:
from sklearn.metrics import f1_score


def train_one_epoch(model, optim, criterion, dataloader, device, writer, epoch):
    model.train()

    train_loss = 0
    all_preds = []
    all_labels = []

    for data, label in dataloader:
        data, label = data.to(device), label.to(device)

        optim.zero_grad()
        output = model(data)
        loss = criterion(output, label)
        loss.backward()
        optim.step()

        train_loss += loss.item()

        preds = output.argmax(dim=1)

        # Move to CPU for sklearn
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(label.cpu().numpy())

    # Compute F1 score
    f1 = f1_score(all_labels, all_preds, average='micro')
    # use 'binary', 'macro', 'weighted' depending on task
    epoch_loss = train_loss / len(dataloader)
    writer.add_scalar(f'Loss/train', epoch_loss, epoch)
    writer.add_scalar(f'F1/train', f1, epoch)

    return epoch_loss, f1

In [ ]:
@torch.no_grad()
def eval_one_epoch(model, criterion, dataloader, device, f1_best, writer, epoch):
    model.eval()
    valid_loss = 0
    all_preds = []
    all_labels = []

    for data, label in dataloader:
        data, label = data.to(device), label.to(device)

        output = model(data)
        loss = criterion(output, label)

        valid_loss += loss.item()

        preds = output.argmax(dim=1)

        # Move to CPU for sklearn
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(label.cpu().numpy())

    # Compute F1 score
    f1 = f1_score(all_labels, all_preds, average='micro')
    # use 'binary', 'macro', 'weighted' depending on task
    if f1 > f1_best:
        # print('Saving..')
        state = {
            'net': model.state_dict(),
            'acc': f1,
            'epoch': epoch,
        }
        if not os.path.isdir('checkpoint'):
            os.mkdir('checkpoint')
        torch.save(state, './checkpoint/ckpt_model.pth')
    epoch_loss = valid_loss / len(dataloader)
    writer.add_scalar(f'Loss/valid', epoch_loss, epoch)
    writer.add_scalar(f'F1/valid', f1, epoch)

    return epoch_loss, f1

In [ ]:
import datetime

train_losses = []
test_losses = []
best_acc = 0
best_epoch = -1
EPOCHS = 10

timestamp = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
log_dir = './runs/simpsons'
writer = SummaryWriter(f'{log_dir}/default_{timestamp}')
for i in range(1, EPOCHS + 1):
    train_loss, train_acc = train_one_epoch(model_simple_cnn, optimizer, criterion, train_loader, DEVICE, writer, i)
    train_losses.append(train_loss)
    test_loss, test_acc = eval_one_epoch(model_simple_cnn, criterion, val_loader, DEVICE, best_acc, writer, i)
    test_losses.append(test_loss)
    best_acc = max(best_acc, test_acc)


In [ ]:
%load_ext tensorboard
%tensorboard --logdir_spec runs:./runs

### Visualize again

In [ ]:
import matplotlib.patches as patches
from matplotlib.font_manager import FontProperties


@torch.no_grad()
def show_images2(n_rows, n_cols, dataset, model):
    fig, axs = plt.subplots(nrows=n_rows, ncols=n_cols, figsize=(n_cols * 4, n_rows * 4), \
                            sharey=True, sharex=True)

    for fig_x in axs.flatten():
        random_characters = int(np.random.uniform(0, len(dataset)))
        im_val, label = dataset[random_characters]
        img_label = " ".join(map(lambda x: x.capitalize(), \
                                 label_encoder.inverse_transform([label])[0].split('_')))
        imshow(im_val.data.cpu(), title=img_label, plt_ax=fig_x)

        actual_text = "Actual : {}".format(img_label)

        font0 = FontProperties()
        font = font0.copy()

        prob_pred = nn.functional.softmax(model(im_val.unsqueeze(0).to(DEVICE)), dim=-1).cpu().numpy()

        predicted_proba = np.max(prob_pred) * 100
        y_pred = np.argmax(prob_pred)

        predicted_label = " ".join(map(lambda x: x.capitalize(), \
                                       label_encoder.inverse_transform([y_pred])[0].split('_')))
        predicted_text = "{}:\n {:.1f}%".format(predicted_label, predicted_proba)

        fig_x.add_patch(patches.Rectangle((0, 190), 7 * len(predicted_label), 25, color='white'))
        fig_x.text(2, 195, predicted_text, horizontalalignment='left', fontproperties=font,
                   verticalalignment='top', fontsize=8, color='black', fontweight='bold')
        fig_x.set_axis_off()

    return None

In [ ]:
show_images2(n_rows=3, n_cols=3, dataset=val_dataset, model=model_simple_cnn)
plt.show()

## Inference

In [ ]:
test_dataset = SimpsonsDataset(test_files, label_encoder=label_encoder, mode="test")
test_loader = DataLoader(test_dataset, shuffle=False, batch_size=64)

In [ ]:
def predict(model, loader):
    model.eval()
    all_predictions = torch.tensor([]).to(DEVICE).int()
    print("Test mode...")
    for inputs in tqdm(loader):
        inputs = inputs.to(DEVICE)

        with torch.no_grad():
            outputs = model(inputs)

            predictions = outputs.argmax(-1).int()
            all_predictions = torch.cat((all_predictions, predictions), 0)
    return all_predictions.cpu()

In [ ]:
predicted_numeric_labels = predict(model_simple_cnn, test_loader)

In [ ]:
predicted_text_labels = label_encoder.inverse_transform(predicted_numeric_labels)

In [ ]:
import pandas as pd

preds = pd.DataFrame({'Id': [path.name for path in test_files], 'Expected': predicted_text_labels})
preds.head(10)

## Adventures?

1. Add augmentations
2. Try another optimizer. i.e. AdamW
3. Transfer Learning
4. Add scheduler https://docs.pytorch.org/docs/stable/generated/torch.optim.lr_scheduler.ReduceLROnPlateau.html, https://docs.pytorch.org/docs/stable/generated/torch.optim.lr_scheduler.CosineAnnealingLR.html
5. Train longer